# Tutorial: оптимизаторы и тестовые функции

Краткая теория и примеры вызова API без Streamlit. Используются модули `core` (тестовые функции, симуляция) и `optimizers` (регистр, создание оптимизатора).

**Запуск:** из корня репозитория, чтобы работали импорты:
```bash
cd AIOptimizersPlayground
jupyter notebook notebooks/OptimizersTutorial.ipynb
```

## 1. Импорты и тестовые функции

`core.get_test_functions()` возвращает словарь: имя функции → `{func, grad}`. Все функции заданы для вектора параметров длины 2 (x, y).

In [ ]:
import asyncio
import numpy as np
import matplotlib.pyplot as plt

from core import get_test_functions, run_optimization as core_run_optimization
from optimizers.registry import create_optimizer, get_optimizer_names

test_functions = get_test_functions()
print("Тестовые функции:", list(test_functions.keys()))
print("Оптимизаторы:", get_optimizer_names())

## 2. Один оптимизатор на Quadratic

Создаём оптимизатор через `create_optimizer(name, lr, momentum, weight_decay, params)`. Для GD/SGD дополнительных параметров нет — передаём пустой словарь. Симуляция — `core.run_optimization` (async), поэтому оборачиваем в `asyncio.run`.

In [ ]:
opt = create_optimizer("GD", lr=0.1, momentum=0.0, wd=0.0, params={})
start = np.array([3.0, -2.0], dtype=np.float64)
trajectory, losses, grad_norms, _ = asyncio.run(
    core_run_optimization(
        opt, start, test_functions, "Quadratic",
        iterations=50, noise_level=0.0, bounds=5.0, add_noise=False
    )
)
print(f"Финальный loss: {losses[-1]:.6f}")
print(f"Траектория: от {trajectory[0]} до {trajectory[-1]}")

## 3. График траектории и Loss

Строим кривую потерь и траекторию в 2D (для Quadratic удобно показать сходимость к (0,0)).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(losses, color="blue")
ax1.set_xlabel("Итерация")
ax1.set_ylabel("Loss")
ax1.set_title("Loss (GD на Quadratic)")
ax1.set_yscale("log")

xy = np.array(trajectory)
ax2.plot(xy[:, 0], xy[:, 1], "b-o", markersize=2)
ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.set_title("Траектория в 2D")
ax2.set_aspect("equal")
plt.tight_layout()
plt.show()

## 4. Сравнение SGD и Adam на Rastrigin

Запускаем два оптимизатора с одной стартовой точки и сравниваем по финальному loss и кривым.

In [ ]:
start = np.array([2.0, 2.0], dtype=np.float64)
results = {}
for name in ["SGD", "Adam"]:
    params = {} if name == "SGD" else {"Adam_beta1": 0.9, "Adam_beta2": 0.999}
    opt = create_optimizer(name, lr=0.01, momentum=0.9, wd=0.0, params=params)
    traj, losses, _, _ = asyncio.run(
        core_run_optimization(
            opt, start.copy(), test_functions, "Rastrigin",
            iterations=150, noise_level=0.0, bounds=5.0, add_noise=False
        )
    )
    results[name] = {"trajectory": traj, "losses": losses}

for name, data in results.items():
    print(f"{name}: финальный loss = {data['losses'][-1]:.4f}")

In [ ]:
plt.figure(figsize=(8, 4))
for name, color in [("SGD", "blue"), ("Adam", "orange")]:
    L = results[name]["losses"]
    plt.plot(L, label=name, color=color)
plt.xlabel("Итерация")
plt.ylabel("Loss")
plt.yscale("log")
plt.legend()
plt.title("SGD vs Adam на Rastrigin")
plt.tight_layout()
plt.show()

## 5. Дальше

- В **Streamlit-приложении** (главная страница, Playground) доступны сценарии, задания, теория и глоссарий.
- Добавить свой оптимизатор: класс в `optimizers/`, регистрация в `optimizers/registry.py`, описание в `core/descriptions.py`.